# 9. Vision LLMs through LiteLLM (Zero-Shot Detection)
Modern Multimodal LLMs (Gemini, Claude, GPT-4o) exhibit spatial reasoning capabilities. We can prompt them to detect fashion items and return their bounding boxes in JSON format.

In this notebook, we:
1. Configure LiteLLM to use Gemini 1.5 Flash (or another vision API).
2. Set up the prompt asking for `[ymin, xmin, ymax, xmax]` normalized coordinates (0-1000).
3. Parse the output JSON.
4. Scale coordinates back to pixel values and visualize.



In [ ]:
import os
import sys

sys.path.append(os.path.abspath("."))

# NOTE: To run this notebook successfully, ensure you have set the appropriate API key environment variable.
# E.g. os.environ["GEMINI_API_KEY"] = "your-api-key"
# We will check if it is set. If not, we will output a warning.
if not os.environ.get("GEMINI_API_KEY"):
    print(
        "WARNING: GEMINI_API_KEY environment variable is not set. API calls will fail."
    )

from fashion_detector.config import Config
from fashion_detector.models.vision_llm import VisionLlmDetector
from fashion_detector.utils import load_image, generate_interactive_html
from IPython.display import HTML

config = Config("config/config.yaml")
detector = VisionLlmDetector(config)

## Execute Vision LLM Object Detection
We query the model zero-shot.



In [ ]:
image = load_image("data/fashion_model_street.jpg")

# Run detection if API key is present, otherwise show mock output
if os.environ.get("GEMINI_API_KEY"):
    detections = detector.detect(image)
else:
    print("API Key not found. Displaying mocked Vision LLM detections...")
    from fashion_detector.models.base import Detection

    # Mocking Gemini 1.5 Flash response structure
    detections = [
        Detection(box=[200.0, 300.0, 650.0, 950.0], label="dress", score=0.95),
        Detection(box=[300.0, 100.0, 500.0, 250.0], label="hat", score=0.90),
        Detection(box=[150.0, 650.0, 350.0, 850.0], label="handbag", score=0.93),
    ]

print(f"Detected {len(detections)} fashion items:")
for d in detections:
    print(f"- {d.label}: score={d.score:.2f}, box={list(map(int, d.box))}")

## Visualizing interactive outputs


In [ ]:
html_str = generate_interactive_html(image, detections, title="Vision LLM Detections")
HTML(html_str)